# `ablation_configs.py` — Ablation & Baseline Experiment Configurations

## Purpose
Generates **config overrides** for each ablation study variant. Instead of duplicating config YAML files, each function takes the base config and returns a **deep copy** with only the relevant parameters changed.

## Design Pattern
Every function returns a list of `(experiment_id, description, config_dict)` tuples — called `ExperimentSpec`. The Experiment Runner (`12_experiment_runner.ipynb`) iterates these to train and evaluate each variant.

## Ablation Studies

| ID | What's being tested | Hypothesis |
|----|---------------------|------------|
| A1 | GCN on/off | Does the dependency graph improve sentiment separation? |
| A2 | Aspect attention vs CLS pooling | Does aspect-guided attention outperform a generic CLS vector? |
| A3 | Loss function variants | Which imbalance-handling loss works best? |
| A4 | With/without synthetic augmentation | Does LLM-generated data help rare classes? |
| A5 | Aspect-specific vs shared classifier head | Do dedicated heads per aspect outperform a shared head? |
| A6 | MSR evaluation with/without GCN | Does the GCN specifically help Mixed Sentiment Resolution? |
| A7 | Hybrid loss weight tuning | What's the best Focal:CB ratio? |

## Baselines (B series)
- B1: Plain RoBERTa (no aspect awareness)
- B2: Full model + vanilla Cross-Entropy
- B3: BERT-base (instead of RoBERTa)
- B4: TF-IDF + LinearSVC (classical ML)
- B5: Flat ABSA — prior-art-style baseline (no GCN, shared head, CE loss)

In [1]:
from tqdm.auto import tqdm
print("⏳ Starting: Loading dependencies and libraries...")
import time
_start_time = time.time()
import copy
from typing import Dict, List, Tuple

# Type alias: each experiment is a (id, description, config_override) triple
ExperimentSpec = Tuple[str, str, dict]
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Loading dependencies and libraries.")


⏳ Starting: Loading dependencies and libraries...
🕒 Done in 0.00s
✅ Completed: Loading dependencies and libraries.


## Aggregator Function

In [2]:
print("⏳ Starting: Defining function get_all_ablation_specs...")
import time
_start_time = time.time()
def get_all_ablation_specs(base_config: dict) -> List[ExperimentSpec]:
    """
    Returns ALL ablation experiment specs by calling each A-series function.
    Pass the base config (loaded from config.yaml) to get fully-formed configs.
    """
    specs = []
    specs.extend(ablation_1_gcn(base_config))
    specs.extend(ablation_2_aspect_attention(base_config))
    specs.extend(ablation_3_loss_function(base_config))
    specs.extend(ablation_4_augmentation(base_config))
    specs.extend(ablation_5_classifier_head(base_config))
    specs.extend(ablation_6_mixed_sentiment(base_config))
    specs.extend(ablation_7_hybrid_weights(base_config))
    return specs
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function get_all_ablation_specs.")


⏳ Starting: Defining function get_all_ablation_specs...
🕒 Done in 0.00s
✅ Completed: Defining function get_all_ablation_specs.


## A1: GCN On/Off

In [3]:
print("⏳ Starting: Defining function ablation_1_gcn...")
import time
_start_time = time.time()
def ablation_1_gcn(base_config: dict) -> List[ExperimentSpec]:
    """
    Tests whether the Dependency GCN improves performance.
    Hypothesis: GCN captures syntactic relationships critical for MSR.
    """
    full   = copy.deepcopy(base_config)                  # Deep copy prevents mutating base_config
    full['experiment']['name'] = 'A1_full_model'         # Add GCN (default)

    no_gcn = copy.deepcopy(base_config)
    no_gcn['model']['use_dependency_gcn']    = False     # Disable GCN module
    no_gcn['data']['use_dependency_parsing'] = False     # Also skip parsing (saves compute)
    no_gcn['experiment']['name']             = 'A1_no_gcn'

    return [
        ('A1_full_model', 'Full model (with Dependency GCN)', full),
        ('A1_no_gcn',     'No GCN — aspect attention only',   no_gcn),
    ]
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function ablation_1_gcn.")


⏳ Starting: Defining function ablation_1_gcn...
🕒 Done in 0.00s
✅ Completed: Defining function ablation_1_gcn.


## A2: Aspect Attention vs CLS Pooling

In [4]:
print("⏳ Starting: Defining function ablation_2_aspect_attention...")
import time
_start_time = time.time()
def ablation_2_aspect_attention(base_config: dict) -> List[ExperimentSpec]:
    attention = copy.deepcopy(base_config)
    attention['experiment']['name'] = 'A2_aspect_attention'       # Aspect-guided MHA (default)

    cls_only = copy.deepcopy(base_config)
    cls_only['model']['use_aspect_attention'] = False             # Fall back to CLS token
    cls_only['experiment']['name']            = 'A2_cls_pooling'

    return [
        ('A2_aspect_attention', 'Aspect-guided MHA attention',     attention),
        ('A2_cls_pooling',      'CLS token pooling (no attention)', cls_only),
    ]
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining function ablation_2_aspect_attention.")


⏳ Starting: Defining function ablation_2_aspect_attention...
🕒 Done in 0.00s
✅ Completed: Defining function ablation_2_aspect_attention.


## A3: Loss Function Variants

In [5]:
print("⏳ Starting: Defining class imbalance...")
import time
_start_time = time.time()
def ablation_3_loss_function(base_config: dict) -> List[ExperimentSpec]:
    """
    Tests each loss component individually and in combination to identify
    which ones contribute to handling class imbalance.
    """
    def make_cfg(name, weights):
        cfg = copy.deepcopy(base_config)
        cfg['training']['loss_weights'] = weights  # Controls Focal/CB/Dice weighting
        cfg['experiment']['name']       = name
        return cfg

    hybrid     = make_cfg('A3_hybrid_loss',  {'focal': 1.0, 'cb': 0.5, 'dice': 0.3})  # All 3
    focal_only = make_cfg('A3_focal_only',   {'focal': 1.0, 'cb': 0.0, 'dice': 0.0})  # Only focal
    cb_only    = make_cfg('A3_cb_only',      {'focal': 0.0, 'cb': 1.0, 'dice': 0.0})  # Only CB
    dice_only  = make_cfg('A3_dice_only',    {'focal': 0.0, 'cb': 0.0, 'dice': 1.0})  # Only dice

    ce = copy.deepcopy(base_config)
    ce['training']['use_ce_loss'] = True   # Flag picked up by experiment_runner to use CrossEntropyLossWrapper
    ce['experiment']['name']      = 'A3_ce_loss'

    return [
        ('A3_hybrid_loss', 'Hybrid Loss (Focal + CB + Dice)', hybrid),
        ('A3_focal_only',  'Focal Loss only',                 focal_only),
        ('A3_cb_only',     'Class-Balanced Loss only',        cb_only),
        ('A3_dice_only',   'Dice Loss only',                  dice_only),
        ('A3_ce_loss',     'Cross-Entropy (no imbalance)',    ce),
    ]
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class imbalance.")


⏳ Starting: Defining class imbalance...
🕒 Done in 0.00s
✅ Completed: Defining class imbalance.


## A4–A7: Augmentation, Classifier Head, MSR, Hybrid Weights

In [6]:
print("⏳ Starting: Defining class performance...")
import time
_start_time = time.time()
def ablation_4_augmentation(base_config):
    """Tests whether LLM-generated synthetic data improves rare-class performance."""
    with_aug = copy.deepcopy(base_config)
    with_aug['data']['train_path'] = 'data/splits/train_augmented.csv'  # Uses augmented set
    with_aug['experiment']['name'] = 'A4_with_augmentation'

    no_aug = copy.deepcopy(base_config)
    no_aug['data']['train_path'] = 'data/splits/train.csv'              # Uses original set
    no_aug['experiment']['name'] = 'A4_no_augmentation'

    return [('A4_with_augmentation', 'With LLM augmentation (10,050 samples)', with_aug),
            ('A4_no_augmentation',   'Without augmentation (9,240 samples)',   no_aug)]


def ablation_5_classifier_head(base_config):
    """Tests 7 aspect-specific heads vs 1 shared head."""
    aspect_specific = copy.deepcopy(base_config)
    aspect_specific['model']['use_shared_classifier'] = False  # Per-aspect heads (default)
    aspect_specific['experiment']['name'] = 'A5_aspect_specific_heads'

    shared = copy.deepcopy(base_config)
    shared['model']['use_shared_classifier'] = True           # One shared head
    shared['experiment']['name'] = 'A5_shared_head'

    return [('A5_aspect_specific_heads', '7 aspect-specific classifier heads', aspect_specific),
            ('A5_shared_head',           'Single shared classifier head',       shared)]


def ablation_6_mixed_sentiment(base_config):
    """
    Both variants use the full MSR evaluator so we can directly compare
    how much the GCN specifically helps handle conflicting sentiments within a review.
    """
    with_gcn = copy.deepcopy(base_config)
    with_gcn['experiment']['name']         = 'A6_msr_with_gcn'
    with_gcn['experiment']['evaluate_msr'] = True  # Trigger MixedSentimentEvaluator post-test

    no_gcn = copy.deepcopy(base_config)
    no_gcn['model']['use_dependency_gcn']    = False
    no_gcn['data']['use_dependency_parsing'] = False
    no_gcn['experiment']['name']             = 'A6_msr_no_gcn'
    no_gcn['experiment']['evaluate_msr']     = True

    return [('A6_msr_with_gcn', 'MSR Eval: Full model + GCN', with_gcn),
            ('A6_msr_no_gcn',   'MSR Eval: No GCN (attention only)', no_gcn)]


def ablation_7_hybrid_weights(base_config):
    """Tunes the Focal:CB ratio without Dice (Dice was dropped after A3)."""
    def make_cfg(name, weights):
        cfg = copy.deepcopy(base_config)
        cfg['training']['loss_weights'] = weights
        cfg['experiment']['name']       = name
        return cfg

    return [
        ('A7_hybrid_cb_05', 'Hybrid: Focal 1.0 + CB 0.5 (no Dice)', make_cfg('A7_hybrid_cb_05', {'focal': 1.0, 'cb': 0.5, 'dice': 0.0})),
        ('A7_hybrid_cb_10', 'Hybrid: Focal 1.0 + CB 1.0 (no Dice)', make_cfg('A7_hybrid_cb_10', {'focal': 1.0, 'cb': 1.0, 'dice': 0.0})),
    ]
print(f"🕒 Done in {time.time() - _start_time:.2f}s")
print("✅ Completed: Defining class performance.")


⏳ Starting: Defining class performance...
🕒 Done in 0.00s
✅ Completed: Defining class performance.


## Quick Test

Shows the generated experiment specs from a minimal base config — confirms the config generators work and shows what `experiment_runner.py` actually iterates over.

In [7]:
import time
_start = time.time()
print("Running ablation_configs quick test...\n")

# Minimal base config — same shape as configs/config.yaml
base_config = {
    'model': {
        'roberta_model': 'roberta-base', 'num_aspects': 7, 'num_classes': 3,
        'hidden_dim': 768, 'dropout': 0.1, 'gcn_layers': 2,
        'use_dependency_gcn': True, 'use_aspect_attention': True,
        'use_shared_classifier': False,
    },
    'data': {
        'train_path': 'data/splits/train_augmented.csv',
        'val_path':   'data/splits/val.csv',
        'test_path':  'data/splits/test.csv',
        'use_dependency_parsing': True,
        'max_seq_length': 128, 'text_column': 'data',
    },
    'training': {
        'loss_weights': {'focal': 1.0, 'cb': 0.5, 'dice': 0.0},
        'batch_size': 16, 'learning_rate': 2e-5, 'num_epochs': 30,
    },
    'aspects': {
        'names': ['stayingpower','texture','smell','price','colour','shipping','packing'],
        'label_map': {'negative': 0, 'neutral': 1, 'positive': 2},
    },
    'experiment': {'name': 'base', 'evaluate_msr': False},
}

# Generate all ablation specs
all_specs = get_all_ablation_specs(base_config)
print(f"Total ablation specs generated: {len(all_specs)}\n")

# Print each experiment's ID and description
for exp_id, desc, cfg in all_specs:
    gcn  = cfg['model'].get('use_dependency_gcn', True)
    attn = cfg['model'].get('use_aspect_attention', True)
    loss = cfg['training'].get('loss_weights', {})
    data = 'augmented' if 'augmented' in cfg['data']['train_path'] else 'original'
    print(f"  {exp_id:<30}  {desc}")

# Check that deep copies don't bleed into base_config
all_specs[0][2]['model']['use_dependency_gcn'] = False
assert base_config['model']['use_dependency_gcn'] == True, "base_config was mutated — deep copy broken!"
print(f"\n[OK] deep copy isolation works — base_config not mutated")

# Show A3 loss configs specifically (most variants)
print("\nA3 loss variants:")
for exp_id, desc, cfg in all_specs:
    if exp_id.startswith('A3'):
        print(f"  {exp_id}: loss_weights = {cfg['training'].get('loss_weights', 'N/A (CE)')}")

print(f"\nAll ablation_configs tests PASSED  ({time.time() - _start:.2f}s)")

Running ablation_configs quick test...

Total ablation specs generated: 17

  A1_full_model                   Full model (with Dependency GCN)
  A1_no_gcn                       No GCN — aspect attention only
  A2_aspect_attention             Aspect-guided MHA attention
  A2_cls_pooling                  CLS token pooling (no attention)
  A3_hybrid_loss                  Hybrid Loss (Focal + CB + Dice)
  A3_focal_only                   Focal Loss only
  A3_cb_only                      Class-Balanced Loss only
  A3_dice_only                    Dice Loss only
  A3_ce_loss                      Cross-Entropy (no imbalance)
  A4_with_augmentation            With LLM augmentation (10,050 samples)
  A4_no_augmentation              Without augmentation (9,240 samples)
  A5_aspect_specific_heads        7 aspect-specific classifier heads
  A5_shared_head                  Single shared classifier head
  A6_msr_with_gcn                 MSR Eval: Full model + GCN
  A6_msr_no_gcn                   MSR 